In [2]:
import json
import time
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

INDEX_URL = "https://www.osha.gov/laws-regs/regulations/standardnumber/1926"
BASE_URL = "https://www.osha.gov"

def create_hardened_session():
    session = requests.Session()
    
    # 1. Advanced headers to mimic a real Windows Chrome browser
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1"
    })
    
    # 2. Add an automated retry strategy if the connection drops unexpectedly
    retries = Retry(
        total=5, 
        backoff_factor=1,  # Waits 1s, 2s, 4s, 8s between retries
        status_forcelist=[500, 502, 503, 504],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    
    return session

def get_regulation_links(session):
    print(f"Scraping index page safely: {INDEX_URL}...")
    try:
        response = session.get(INDEX_URL, timeout=15)
        if response.status_code != 200:
            print(f"Failed to fetch index page. Code: {response.status_code}")
            return []

        soup = BeautifulSoup(response.text, 'html.parser')
        sections_data = []

        for link in soup.find_all('a', href=True):
            href = link['href']
            if "/laws-regs/regulations/standardnumber/1926/1926." in href:
                full_url = BASE_URL + href if href.startswith('/') else href
                section_title = link.get_text(strip=True)
                
                sections_data.append({
                    "section_id": section_title.split('-')[0].strip(),
                    "title": section_title,
                    "url": full_url
                })
                
        print(f"Found {len(sections_data)} regulation sections.")
        return sections_data
    except Exception as e:
        print(f"Network error on index crawl: {e}")
        return []

def scrape_section_content(session, url):
    try:
        # Pass through the authenticated session
        response = session.get(url, timeout=15)
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.text, 'html.parser')
        content_div = soup.find('div', {'class': 'main-content'}) or soup.find('article') or soup.find('div', {'id': 'block-mainpagecontent'})
        
        if not content_div:
            content_div = soup.find('body')

        paragraphs = []
        for element in content_div.find_all(['p', 'li']):
            text = element.get_text(strip=True)
            if text and len(text) > 15: 
                paragraphs.append(text)
                
        return "\n\n".join(paragraphs)
    except Exception as e:
        print(f"Skipping page due to network hiccup on {url}: {str(e)}")
        return None

def main():
    # Initialize our safe browser session
    session = create_hardened_session()
    
    regulation_targets = get_regulation_links(session)
    if not regulation_targets:
        print("No targets found. The firewall blocked the initial index step.")
        return

    scraped_dataset = []
    

    for target in regulation_targets[:374]: 
        print(f"Scraping text for: {target['title']}")
        
        text_content = scrape_section_content(session, target['url'])
        if text_content:
            target['full_text'] = text_content
            scraped_dataset.append(target)
            
        # VERY IMPORTANT: Keep this 2-second buffer so we don't look like a DDOS attack
        time.sleep(2)
        
    output_filename = "osha_raw_documents.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(scraped_dataset, f, indent=4, ensure_ascii=False)
        
    print(f"\nSuccess! Data successfully saved to {output_filename}")

if __name__ == "__main__":
    main()

Scraping index page safely: https://www.osha.gov/laws-regs/regulations/standardnumber/1926...
Found 374 regulation sections.
Scraping text for: 1926.1 - Purpose and scope.
Scraping text for: 1926.2 - Variances from safety and health standards.
Scraping text for: 1926.3 - Inspections - right of entry.
Scraping text for: 1926.4 - Rules of practice for administrative adjudications for enforcement of safety and health standards.
Scraping text for: 1926.5 - OMB control numbers under the Paperwork Reduction Act.
Scraping text for: 1926.6 - Incorporation by reference.
Scraping text for: 1926.10 - Scope of subpart.
Scraping text for: 1926.11 - Coverage under section 103 of the act distinguished.
Scraping text for: 1926.12 - Reorganization Plan No. 14 of 1950.
Scraping text for: 1926.13 - Interpretation of statutory terms.
Scraping text for: 1926.14 - Federal contract for "mixed" types of performance.
Scraping text for: 1926.15 - Relationship to the Service Contract Act; Walsh-Healey Public Con

In [9]:
!py -m pip uninstall langgraph langgraph-checkpoint langgraph-sdk langchain langchain-core langchain-community -y

Found existing installation: langgraph 1.2.2
Uninstalling langgraph-1.2.2:
  Successfully uninstalled langgraph-1.2.2
Found existing installation: langgraph-checkpoint 4.1.0
Uninstalling langgraph-checkpoint-4.1.0:
  Successfully uninstalled langgraph-checkpoint-4.1.0
Found existing installation: langgraph-sdk 0.3.3
Uninstalling langgraph-sdk-0.3.3:
  Successfully uninstalled langgraph-sdk-0.3.3
Found existing installation: langchain 1.3.2
Uninstalling langchain-1.3.2:
  Successfully uninstalled langchain-1.3.2
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2


In [10]:
!py -m pip install langchain langchain-core langchain-community langgraph

  Using cached langchain-1.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langgraph-1.2.2-py3-none-any.whl.metadata (8.0 kB)
Using cached langchain-1.3.2-py3-none-any.whl (121 kB)
Using cached langchain_core-1.4.0-py3-none-any.whl (548 kB)
Using cached langgraph-1.2.2-py3-none-any.whl (236 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)

   ---------------------------------------- 0/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ ---------------------

In [13]:
!py -m pip install --upgrade langchain==0.3.14 langchain-core==0.3.29 langchain-community==0.3.14 langgraph==0.2.60

INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 2.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.5 MB 12.8 MB/s eta 0:00:01
   ---------------- ----------------------- 1.0/2.5 MB 3.5 MB/s eta 0:00:01
   -------------------- ------------------- 1.3/2.5 MB 3.0 MB/s eta 0:00:01
   ----------------------------- ---------- 1.8/2.5 MB 2.7 MB/s eta 0:00:01
   ------------------------------------- -- 2.4/2.5 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 2.4 MB/s  0:00:01

  Attempting uninstall: lang

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.3.29 which is incompatible.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.29 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.5 which is incompatible.
langchain-google-genai 4.2.2 requires langchain-core<2.0.0,>=1.2.29, but you have langchain-core 0.3.29 which is incompatible.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have lang

In [14]:
! py -m pip install requests beautifulsoup4 langchain_core langchain_text_splitters langchain_community chromadb fastembed langchain_storage

  Using cached fastembed-0.8.0-py3-none-any.whl.metadata (10 kB)


ERROR: Ignored the following versions that require a different python version: 0.0.1 Requires-Python >=3.8.0,<3.12; 0.0.1a1 Requires-Python >=3.8.0,<3.12; 0.0.1a2 Requires-Python >=3.8.0,<3.12; 0.0.1a3 Requires-Python >=3.8.0,<3.12; 0.0.2 Requires-Python >=3.8.0,<3.12; 0.0.2a0 Requires-Python >=3.8.0,<3.12; 0.0.3 Requires-Python >=3.8.0,<3.12; 0.0.3a1 Requires-Python >=3.8.0,<3.12; 0.0.4 Requires-Python >=3.8.0,<3.12; 0.0.5 Requires-Python >=3.8.0,<3.12; 0.0.5a1 Requires-Python >=3.8.0,<3.12; 0.0.5a2 Requires-Python >=3.8.0,<3.12; 0.1.0 Requires-Python >=3.8.0,<3.12; 0.1.1 Requires-Python >=3.8.0,<3.12; 0.1.2 Requires-Python >=3.8.0,<3.12; 0.1.3 Requires-Python >=3.8.0,<3.12; 0.2.0 Requires-Python >=3.8.0,<3.13; 0.2.1 Requires-Python >=3.8.0,<3.13; 0.2.2 Requires-Python >=3.8.0,<3.13; 0.2.3 Requires-Python >=3.8.0,<3.13; 0.2.4 Requires-Python >=3.8.0,<3.13; 0.2.5 Requires-Python >=3.8.0,<3.13; 0.2.6 Requires-Python >=3.8.0,<3.13; 0.2.7 Requires-Python >=3.8.0,<3.13; 0.3.0 Requires-Pyth

In [15]:
!py -m pip uninstall langchain-core -y


Found existing installation: langchain-core 0.3.29
Uninstalling langchain-core-0.3.29:
  Successfully uninstalled langchain-core-0.3.29


In [16]:
!py -m pip install langchain-core==0.3.29 --no-deps

  Using cached langchain_core-0.3.29-py3-none-any.whl.metadata (6.3 kB)
Using cached langchain_core-0.3.29-py3-none-any.whl (411 kB)


In [1]:
import sys
import importlib

# 1. Clear out any stale import caches
importlib.invalidate_caches()

print("--- EXECUTABLE & PATH DEBUG ---")
print(f"Active Python Executable: {sys.executable}")
print("Python Search Paths:")
for path in sys.path:
    if "site-packages" in path:
        print(f" -> {path}")
print("--------------------------------\n")


--- EXECUTABLE & PATH DEBUG ---
Active Python Executable: c:\Users\romma\AppData\Local\Programs\Python\Python313\python.exe
Python Search Paths:
 -> C:\Users\romma\AppData\Roaming\Python\Python313\site-packages
 -> C:\Users\romma\AppData\Roaming\Python\Python313\site-packages\win32
 -> C:\Users\romma\AppData\Roaming\Python\Python313\site-packages\win32\lib
 -> C:\Users\romma\AppData\Roaming\Python\Python313\site-packages\Pythonwin
 -> c:\Users\romma\AppData\Local\Programs\Python\Python313\Lib\site-packages
--------------------------------



In [2]:
import os
from gptcache import cache
from gptcache.adapter.api import init_similar_cache
from gptcache.processor.pre import get_prompt
from gptcache.embedding import Onnx  # 🚀 Added fast local embedding driver
from langchain_community.cache import GPTCache
from langchain_core.globals import set_llm_cache

# 1. Ensure the persistence directory exists
cache_directory = "merged_cache_db"
if not os.path.exists(cache_directory):
    os.makedirs(cache_directory)

# 2. Initialize the semantic cache parameters cleanly
# Fixed: swapped 'pre_processor' -> 'pre_func' and passed an Onnx encoder instance
init_similar_cache(
    cache_obj=cache,
    pre_func=get_prompt,               # 💡 Correct parameter name
    embedding=Onnx(),                  # 💡 Creates a fast local text embedder 
    data_dir=cache_directory
)

# 3. Register it globally into LangChain's execution engine
gpt_cache = GPTCache()
set_llm_cache(gpt_cache)

print("⚡ [Cache Layer Initialized]: Semantic cache VDB successfully mapped to storage.")

c:\Users\romma\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\romma\.cache\huggingface\hub\models--GPTCache--paraphrase-albert-small-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
2026-06-01 16:08:51,444 - 17624 - _http.py-_http:904 - WARNING: Warning: You a

start to install package: faiss-cpu


PipInstallError: Ran into error installing faiss-cpu.